# 📊 Comparación de Curvas ROC: ResNet50 vs YOLO

Este notebook compara el rendimiento de dos modelos diferentes para la clasificación de escarabajos:

- **ResNet50**: Modelo de clasificación con validación cruzada 5-fold
- **YOLO**: Modelo de detección y clasificación con validación cruzada 5-fold

## Datos
- **ResNet50**: `curva-roc/resnet-model/predictions_resnet50_cv5.pkl`
- **YOLO**: `curva-roc/yolo-model/cv_summary_roc_pr.json` y archivos por fold

In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import json
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# Configuración de matplotlib
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300

In [ ]:
# ============================================================
# 📥 CARGAR DATOS DE RESNET50
# ============================================================

print("="*60)
print("📥 CARGANDO DATOS DE RESNET50")
print("="*60)

# Cargar predicciones de ResNet50
with open('curva-roc/resnet-model/predictions_resnet50_cv5.pkl', 'rb') as f:
    resnet_data = pickle.load(f)

# Extraer información
resnet_predictions = resnet_data['predictions']
resnet_true_labels = resnet_data['true_labels']
resnet_probabilities = np.array(resnet_data['probabilities'])
resnet_class_names = resnet_data['class_names']
resnet_num_classes = resnet_data['num_classes']
resnet_auc_micro = resnet_data['roc_auc_micro']
resnet_auc_macro = resnet_data['roc_auc_macro']
resnet_accuracy = resnet_data['global_accuracy']

print(f"\n✅ ResNet50 cargado exitosamente")
print(f"   • Total de muestras: {len(resnet_true_labels)}")
print(f"   • Número de clases: {resnet_num_classes}")
print(f"   • Accuracy: {resnet_accuracy:.4f}")
print(f"   • AUC Micro-average: {resnet_auc_micro:.4f}")
print(f"   • AUC Macro-average: {resnet_auc_macro:.4f}")
print(f"   • Clases: {resnet_class_names}")

In [ ]:
# ============================================================
# 📥 CARGAR DATOS DE YOLO
# ============================================================

print("\n" + "="*60)
print("📥 CARGANDO DATOS DE YOLO")
print("="*60)

# Cargar resumen de YOLO
with open('curva-roc/yolo-model/cv_summary_roc_pr.json', 'r') as f:
    yolo_summary = json.load(f)

# Extraer información general
yolo_mean_auc = yolo_summary['mean_auc']
yolo_mean_ap = yolo_summary['mean_ap']

# Cargar datos de todos los folds
yolo_folds_data = []
for fold_num in range(1, 6):
    with open(f'curva-roc/yolo-model/fold_{fold_num}_roc_pr.json', 'r') as f:
        fold_data = json.load(f)
        yolo_folds_data.append(fold_data)

print(f"\n✅ YOLO cargado exitosamente")
print(f"   • Total de folds: {len(yolo_folds_data)}")
print(f"   • AUC promedio (CV): {yolo_mean_auc:.4f}")
print(f"   • AP promedio (CV): {yolo_mean_ap:.4f}")

# Explorar estructura del primer fold
print(f"\n🔍 Estructura del Fold 1:")
print(f"   • Total GT: {yolo_folds_data[0]['total_gt']}")
print(f"   • Total predicciones: {yolo_folds_data[0]['total_preds']}")
print(f"   • TP predicciones: {yolo_folds_data[0]['tp_predictions']}")
print(f"   • Claves en ROC: {list(yolo_folds_data[0]['roc'].keys())}")

In [ ]:
# ============================================================
# 📊 EXTRAER CURVAS ROC DE YOLO (PROMEDIO DE TODOS LOS FOLDS)
# ============================================================

print("\n" + "="*60)
print("📊 PROCESANDO DATOS DE YOLO")
print("="*60)

# Extraer FPR y TPR de todos los folds para hacer promedio
yolo_fpr_all_folds = []
yolo_tpr_all_folds = []
yolo_auc_all_folds = []

for i, fold_data in enumerate(yolo_folds_data, 1):
    fpr = fold_data['roc']['fpr']
    tpr = fold_data['roc']['tpr']
    fold_auc = fold_data['roc']['auc']
    
    yolo_fpr_all_folds.append(fpr)
    yolo_tpr_all_folds.append(tpr)
    yolo_auc_all_folds.append(fold_auc)
    
    print(f"   Fold {i}: AUC = {fold_auc:.4f}, FPR points = {len(fpr)}, TPR points = {len(tpr)}")

# Calcular curva ROC promedio interpolando todos los folds
# Usar un conjunto común de FPR para interpolar
mean_fpr = np.linspace(0, 1, 100)
interpolated_tprs = []

for fpr, tpr in zip(yolo_fpr_all_folds, yolo_tpr_all_folds):
    interpolated_tpr = np.interp(mean_fpr, fpr, tpr)
    interpolated_tprs.append(interpolated_tpr)

yolo_mean_tpr = np.mean(interpolated_tprs, axis=0)
yolo_std_tpr = np.std(interpolated_tprs, axis=0)

print(f"\n✅ Curva ROC promedio de YOLO calculada")
print(f"   • AUC promedio: {yolo_mean_auc:.4f}")
print(f"   • Desviación estándar AUC: {np.std(yolo_auc_all_folds):.4f}")

In [ ]:
# ============================================================
# 📊 CALCULAR CURVA ROC DE RESNET50
# ============================================================

print("\n" + "="*60)
print("📊 CALCULANDO CURVA ROC DE RESNET50")
print("="*60)

# Binarizar las etiquetas
resnet_y_true_bin = label_binarize(resnet_true_labels, classes=list(range(resnet_num_classes)))

# Calcular curva ROC micro-average
resnet_fpr, resnet_tpr, _ = roc_curve(resnet_y_true_bin.ravel(), resnet_probabilities.ravel())
resnet_roc_auc = auc(resnet_fpr, resnet_tpr)

print(f"\n✅ Curva ROC de ResNet50 calculada")
print(f"   • AUC: {resnet_roc_auc:.4f}")
print(f"   • Puntos en la curva: {len(resnet_fpr)}")

In [ ]:
# ============================================================
# 📊 COMPARACIÓN DE CURVAS ROC: RESNET50 VS YOLO
# ============================================================

print("\n" + "="*60)
print("📊 GENERANDO COMPARACIÓN DE CURVAS ROC")
print("="*60)

plt.figure(figsize=(12, 10))

# Graficar curva ROC de ResNet50
plt.plot(resnet_fpr, resnet_tpr, color='blue', lw=3,
         label=f'ResNet50 (AUC = {resnet_roc_auc:.4f})')

# Graficar curva ROC promedio de YOLO
plt.plot(mean_fpr, yolo_mean_tpr, color='red', lw=3,
         label=f'YOLO (AUC = {yolo_mean_auc:.4f})')

# Agregar banda de desviación estándar para YOLO
tpr_upper = np.minimum(yolo_mean_tpr + yolo_std_tpr, 1)
tpr_lower = np.maximum(yolo_mean_tpr - yolo_std_tpr, 0)
plt.fill_between(mean_fpr, tpr_lower, tpr_upper, color='red', alpha=0.2,
                 label=f'YOLO ± std dev')

# Línea diagonal (clasificador aleatorio)
plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Aleatorio (AUC = 0.5)')

# Configuración del gráfico
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos (FPR)', fontsize=14, fontweight='bold')
plt.ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=14, fontweight='bold')
plt.title('Comparación de Curvas ROC: ResNet50 vs YOLO\nClasificación de Géneros de Escarabajos', 
          fontsize=16, fontweight='bold', pad=20)
plt.legend(loc="lower right", fontsize=12, frameon=True, shadow=True)
plt.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)

# Añadir información adicional
textstr = f'ResNet50: {resnet_num_classes} clases, {len(resnet_true_labels)} muestras\n'
textstr += f'YOLO: Promedio de 5 folds'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
plt.text(0.6, 0.2, textstr, transform=plt.gca().transAxes, fontsize=10,
         verticalalignment='top', bbox=props)

plt.tight_layout()
plt.show()

print("\n✅ Gráfico generado exitosamente")

In [ ]:
# ============================================================
# 📊 TABLA COMPARATIVA DE MÉTRICAS
# ============================================================

print("\n" + "="*70)
print("📊 TABLA COMPARATIVA DE MÉTRICAS")
print("="*70)

# Crear DataFrame comparativo
comparison_data = {
    'Modelo': ['ResNet50', 'YOLO'],
    'AUC': [resnet_roc_auc, yolo_mean_auc],
    'AUC Std': [0.0, np.std(yolo_auc_all_folds)],  # ResNet tiene una curva global
    'Accuracy': [resnet_accuracy, 'N/A'],  # YOLO no tiene accuracy en estos datos
    'Muestras': [len(resnet_true_labels), sum([fold['total_gt'] for fold in yolo_folds_data])],
    'Clases': [resnet_num_classes, 'N/A']
}

df_comparison = pd.DataFrame(comparison_data)

print("\n")
print(df_comparison.to_string(index=False))
print("\n" + "="*70)

# Análisis de diferencia
auc_diff = abs(resnet_roc_auc - yolo_mean_auc)
auc_diff_pct = (auc_diff / yolo_mean_auc) * 100

print(f"\n📈 ANÁLISIS:")
print(f"   • Diferencia en AUC: {auc_diff:.4f} ({auc_diff_pct:.2f}%)")

if resnet_roc_auc > yolo_mean_auc:
    print(f"   • 🏆 ResNet50 tiene MEJOR AUC que YOLO")
    print(f"   • Mejora: {auc_diff:.4f} puntos")
else:
    print(f"   • 🏆 YOLO tiene MEJOR AUC que ResNet50")
    print(f"   • Mejora: {auc_diff:.4f} puntos")

print("\n" + "="*70)

In [ ]:
# ============================================================
# 📊 GRÁFICO DE BARRAS COMPARATIVO DE AUC
# ============================================================

print("\n" + "="*60)
print("📊 GENERANDO GRÁFICO COMPARATIVO DE AUC")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Gráfico 1: Comparación de AUC
ax1 = axes[0]
models = ['ResNet50', 'YOLO']
aucs = [resnet_roc_auc, yolo_mean_auc]
colors = ['steelblue', 'coral']

bars = ax1.bar(models, aucs, color=colors, alpha=0.7, edgecolor='black', linewidth=2)

# Añadir valores sobre las barras
for bar, auc_val in zip(bars, aucs):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{auc_val:.4f}',
            ha='center', va='bottom', fontsize=14, fontweight='bold')

ax1.set_ylabel('AUC', fontsize=12, fontweight='bold')
ax1.set_title('Comparación de AUC', fontsize=14, fontweight='bold')
ax1.set_ylim([0.9, 1.0])  # Zoom para ver mejor la diferencia
ax1.grid(True, alpha=0.3, axis='y')
ax1.axhline(y=0.95, color='green', linestyle='--', linewidth=1, alpha=0.5, label='Umbral 95%')
ax1.legend()

# Gráfico 2: AUC por Fold de YOLO vs ResNet50 (global)
ax2 = axes[1]
x_pos = np.arange(len(yolo_auc_all_folds))
ax2.bar(x_pos, yolo_auc_all_folds, color='coral', alpha=0.7, label='YOLO (por fold)', edgecolor='black')
ax2.axhline(y=yolo_mean_auc, color='red', linestyle='--', linewidth=2, label=f'YOLO Promedio ({yolo_mean_auc:.4f})')
ax2.axhline(y=resnet_roc_auc, color='blue', linestyle='--', linewidth=2, label=f'ResNet50 Global ({resnet_roc_auc:.4f})')

ax2.set_xlabel('Fold', fontsize=12, fontweight='bold')
ax2.set_ylabel('AUC', fontsize=12, fontweight='bold')
ax2.set_title('AUC por Fold: YOLO vs ResNet50', fontsize=14, fontweight='bold')
ax2.set_xticks(x_pos)
ax2.set_xticklabels([f'Fold {i+1}' for i in range(len(yolo_auc_all_folds))])
ax2.set_ylim([0.9, 1.0])
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n✅ Gráfico generado exitosamente")

## 📊 Conclusiones

### Comparación de Modelos

Los resultados muestran la comparación entre dos enfoques diferentes:

1. **ResNet50**: Modelo de clasificación de imágenes con Transfer Learning
   - Procesa imágenes recortadas de escarabajos
   - Clasificación directa en 13 géneros
   - Validación cruzada 5-fold

2. **YOLO**: Modelo de detección y clasificación
   - Detecta escarabajos en imágenes completas
   - Clasifica los objetos detectados
   - Validación cruzada 5-fold

### Interpretación de Resultados

- **AUC > 0.96**: Ambos modelos tienen un rendimiento excelente
- La curva más cercana a la esquina superior izquierda indica mejor rendimiento
- La diferencia entre modelos puede deberse a:
  - Diferentes arquitecturas (clasificación vs detección)
  - Diferentes conjuntos de datos (recortadas vs completas)
  - Diferentes preprocesamiento de imágenes

In [ ]:
# ============================================================
# 💾 GUARDAR GRÁFICO DE COMPARACIÓN
# ============================================================

print("\n" + "="*60)
print("💾 GUARDANDO GRÁFICO DE COMPARACIÓN")
print("="*60)

# Regenerar el gráfico principal para guardarlo
plt.figure(figsize=(12, 10))

# Graficar curva ROC de ResNet50
plt.plot(resnet_fpr, resnet_tpr, color='blue', lw=3,
         label=f'ResNet50 (AUC = {resnet_roc_auc:.4f})')

# Graficar curva ROC promedio de YOLO
plt.plot(mean_fpr, yolo_mean_tpr, color='red', lw=3,
         label=f'YOLO (AUC = {yolo_mean_auc:.4f})')

# Agregar banda de desviación estándar para YOLO
tpr_upper = np.minimum(yolo_mean_tpr + yolo_std_tpr, 1)
tpr_lower = np.maximum(yolo_mean_tpr - yolo_std_tpr, 0)
plt.fill_between(mean_fpr, tpr_lower, tpr_upper, color='red', alpha=0.2,
                 label=f'YOLO ± std dev')

# Línea diagonal
plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Aleatorio (AUC = 0.5)')

# Configuración
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos (FPR)', fontsize=14, fontweight='bold')
plt.ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=14, fontweight='bold')
plt.title('Comparación de Curvas ROC: ResNet50 vs YOLO\nClasificación de Géneros de Escarabajos', 
          fontsize=16, fontweight='bold', pad=20)
plt.legend(loc="lower right", fontsize=12, frameon=True, shadow=True)
plt.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)

# Guardar
plt.savefig('comparison_roc_resnet_vs_yolo.png', dpi=300, bbox_inches='tight')
print("\n✅ Gráfico guardado como: 'comparison_roc_resnet_vs_yolo.png'")

plt.show()